# Convert CSR → cuGraph edgelist

In [14]:
import cugraph
import cudf
from scipy.sparse import csr_matrix

def csr_to_cugraph(csr_mat):
    coo = csr_mat.tocoo()
    edges = cudf.DataFrame({
        "src": coo.row,
        "dst": coo.col,
        "weight": coo.data
    })
    G = cugraph.Graph()
    G.from_cudf_edgelist(edges, source='src', destination='dst', edge_attr='weight', renumber=False)
    return G

Input: CSR matrix of WNN graph

Output: cugraph.Graph object ready for clustering

# Run Leiden / Louvain clustering

In [15]:
def run_gpu_clustering(G, method="leiden", resolution=1.0, random_state=0):
    """
    Run community detection on GPU WNN graph
    """
    if method.lower() == "leiden":
        import cugraph.community as community
        cluster_labels = community.leiden(G, resolution=resolution, random_state=random_state)
    elif method.lower() == "louvain":
        import cugraph.community as community
        cluster_labels = community.louvain(G)
    else:
        raise ValueError("Method must be 'leiden' or 'louvain'")
    
    return cluster_labels

Input: cugraph.Graph

Output: cudf.DataFrame with vertex (cell index) and partition (cluster label)

# Map clusters back to obs_names

In [16]:
def map_clusters_to_obs(adata, cluster_df, obs_key="wnn_leiden"):
    """
    Save cluster assignments into adata.obs
    """
    cluster_pd = cluster_df.to_pandas().set_index("vertex")
    adata.obs[obs_key] = cluster_pd['partition'].astype(str)

Makes clusters visible in adata.obs for plotting and downstream analysis

# QC cluster outputs

In [17]:
def qc_clusters(adata, obs_key="wnn_leiden"):
    n_clusters = adata.obs[obs_key].nunique()
    print(f"Number of clusters: {n_clusters}")
    print("Cluster sizes:")
    print(adata.obs[obs_key].value_counts())

Optional: run the clustering multiple times to check determinism

# Unit tests

In [18]:
def test_clusters(adata, cluster_df):
    assert len(cluster_df) == adata.n_obs, "Cluster labels do not match number of cells"

In [19]:
# ===== Person 4: WNN Clustering (Leiden/Louvain) =====
import cudf
import cugraph
from scipy.sparse import csr_matrix

# --- 1️⃣ Convert CSR → cuGraph Graph ---
def csr_to_cugraph(csr_mat):
    coo = csr_mat.tocoo()
    edges = cudf.DataFrame({
        "src": coo.row,
        "dst": coo.col,
        "weight": coo.data
    })
    G = cugraph.Graph()
    G.from_cudf_edgelist(edges, source='src', destination='dst', edge_attr='weight', renumber=False)
    return G

# --- 2️⃣ Run GPU Clustering ---
def run_gpu_clustering(G, method="leiden", resolution=1.0, random_state=0):
    import cugraph.community as community
    if method.lower() == "leiden":
        cluster_labels = community.leiden(G, resolution=resolution, random_state=random_state)
    elif method.lower() == "louvain":
        cluster_labels = community.louvain(G)
    else:
        raise ValueError("Method must be 'leiden' or 'louvain'")
    return cluster_labels

# --- 3️⃣ Map clusters back to adata.obs ---
def map_clusters_to_obs(adata, cluster_df, obs_key="wnn_leiden"):
    cluster_pd = cluster_df.to_pandas().set_index("vertex")
    adata.obs[obs_key] = cluster_pd['partition'].astype(str)

# --- 4️⃣ QC cluster outputs ---
def qc_clusters(adata, obs_key="wnn_leiden"):
    n_clusters = adata.obs[obs_key].nunique()
    print(f"Number of clusters: {n_clusters}")
    print("Cluster sizes:")
    print(adata.obs[obs_key].value_counts())

# --- 5️⃣ Run everything ---
# Make sure your fused WNN CSR matrix is stored in adata.obsp["wnn_connectivities"]
G = csr_to_cugraph(adata.obsp["wnn_connectivities"])
clusters = run_gpu_clustering(G, method="leiden", resolution=1.0, random_state=0)
map_clusters_to_obs(adata, clusters, obs_key="wnn_leiden")
qc_clusters(adata, obs_key="wnn_leiden")


NameError: name 'adata' is not defined